In [26]:
tl_model.load_weights("tl_best.keras")      

In [42]:
base_model.trainable = True
fine_tune_at = 100                       

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False              

trainable = sum(int(np.prod(w.shape)) for w in tl_model.trainable_weights)
print("trainable params:", trainable)        

trainable params: 1844740


In [43]:
tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [44]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("ft_best.keras", monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.BackupAndRestore("backup_ft"),
]

start = time.time()
history_ft = tl_model.fit(train_ds, validation_data=val_ds, epochs=15,
                          class_weight=class_weight, callbacks=callbacks)
ft_time = time.time() - start
print(f"Training time: {ft_time:.0f}s")

Epoch 1/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 90s 225ms/step - accuracy: 0.9734 - loss: 0.0747 - val_accuracy: 0.9453 - val_loss: 0.1571
Epoch 2/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 19s 112ms/step - accuracy: 0.9769 - loss: 0.0634 - val_accuracy: 0.9542 - val_loss: 0.1270
Epoch 3/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 21s 122ms/step - accuracy: 0.9785 - loss: 0.0581 - val_accuracy: 0.9542 - val_loss: 0.1597
Epoch 4/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 37s 95ms/step - accuracy: 0.9823 - loss: 0.0533 - val_accuracy: 0.9554 - val_loss: 0.1547
Epoch 5/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 16s 95ms/step - accuracy: 0.9836 - loss: 0.0485 - val_accuracy: 0.9475 - val_loss: 0.1810
Epoch 6/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 17s 104ms/step - accuracy: 0.9846 - loss: 0.0467 - val_accuracy: 0.9531 - val_loss: 0.1423
Epoch 7/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 17s 99ms/step - accuracy: 0.9862 - loss: 0.0428 - val_accuracy: 0.9386 - val_loss: 0.1926
Training time: 217s


In [34]:
test_loss_tl, test_acc_tl = tl_model.evaluate(test_ds)
y_pred_tl = np.argmax(tl_model.predict(test_ds), axis=1)

print("test accuracy:", test_acc_tl)

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9497 - loss: 0.1304
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step
test accuracy: 0.9497153759002686


In [35]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_tl, target_names=CLASSES))

              precision    recall  f1-score   support

      glioma       0.99      0.93      0.96       243
  meningioma       0.92      0.89      0.90       247
     notumor       0.99      0.97      0.98       300
   pituitary       0.90      1.00      0.95       264

    accuracy                           0.95      1054
   macro avg       0.95      0.95      0.95      1054
weighted avg       0.95      0.95      0.95      1054



In [36]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_tl)
print(cm)

[[227  13   0   3]
 [  2 219   2  24]
 [  1   5 292   2]
 [  0   1   0 263]]


In [37]:
notumor = CLASSES.index("notumor")
missed = ((y_test != notumor) & (y_pred_tl == notumor)).sum()
print(missed, "tumours predicted as notumor, out of", (y_test != notumor).sum())

2 tumours predicted as notumor, out of 754
